# 08 CV Frontier：前沿回顾——自监督、评估指标与展望

> 前置：本模块全部课程。
> 目标：把"识别"与"生成"两条线的现代进展收束成一张全景图，并实现一个 FID 直觉版指标来理解**如何评估生成模型**。

## 1. 识别线：从监督到自监督

监督学习需要海量标注。**自监督（self-supervised）**让模型从数据本身造"标签"：

- **对比学习（SimCLR / MoCo）**：同一张图的两个增强版本（裁剪/变色/模糊）应拉近，不同图应拉远——学出与任务无关的通用特征。核心是 InfoNCE 损失。
- **MAE（掩码自编码器）**：把 75% 的 patch 盖住，让 ViT 重建它们——"完形填空"式学习。不用负样本，简单有效。
- **CLIP**：图像-文本对比学习。图与对应描述拉近——**一个模型同时学会视觉与语言**，成为文生图/多模态的底座。

共同点：**预训练目标 ≠ 下游任务**，但学到的特征迁移性极强，这正是 02 课"迁移学习"的极致形态。

## 2. 生成线：范式竞争与合流

| 范式 | 代表 | 优点 | 缺点 |
|---|---|---|---|
| GAN | StyleGAN | 采样快、质量高 | 训练不稳、模式坍缩 |
| VAE | VQ-VAE | 稳定、潜空间可编辑 | 生成偏糊 |
| 扩散 | DDPM / SD / Flux | 质量最高、条件灵活 | 采样慢 |
| 流/一致性 | Rectified Flow / LCM | 采样快、可逼近扩散质量 | 生态较新 |

趋势：**扩散与流模型成为默认选择**；GAN 退居视频/实时；VAE 退居压缩前端。

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


PyTorch version: 2.13.0


## 3. 评估生成模型：FID 的直觉

生成质量不能只看"像不像"。**FID（Fréchet Inception Distance）**的做法：

1. 用 Inception 网络把真实图和生成图各提成特征分布；
2. 假设两组特征是高斯分布，算它们之间的 **Wasserstein-2 距离**（用均值 + 协方差刻画）：

$$\text{FID} = \|\mu_r - \mu_g\|^2 + \operatorname{tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r\Sigma_g)^{1/2}\right)$$

**FID 越低越好**。下面用随机特征模拟"好/坏"生成器，验证这个指标能正确排序。

In [2]:
def sqrtm(A):
    w, V = np.linalg.eigh(A)          # 特征分解实现矩阵平方根
    return (V * np.sqrt(np.clip(w, 0, None))) @ V.T

def fid_ish(real, gen):
    mu_r, mu_g = real.mean(0), gen.mean(0)
    cov_r, cov_g = np.cov(real.T), np.cov(gen.T)
    m = ((mu_r - mu_g) ** 2).sum()
    tr = np.trace(cov_r + cov_g - 2 * sqrtm(cov_r @ cov_g).real)
    return m + tr

rng = np.random.default_rng(0)
real = rng.standard_normal((1000, 64))
gen_good = real + rng.standard_normal((1000, 64)) * 0.05     # 与真实几乎一致
gen_ok   = real + rng.standard_normal((1000, 64)) * 0.5      # 有噪声偏差
gen_bad  = rng.standard_normal((1000, 64)) + 0.8             # 完全不同分布

for name, g in [("好生成器", gen_good), ("一般生成器", gen_ok), ("坏生成器", gen_bad)]:
    print(f"{name}: FID ≈ {fid_ish(real, g):.4f}")

好生成器: FID ≈ 0.0074
一般生成器: FID ≈ 1.4076
坏生成器: FID ≈ 43.1650


> 观察：FID 把"好/一般/坏"严格排序。它同时惩罚**分布偏移（均值项）**和**多样性丢失（协方差项）**——这也是为什么 FID 比单纯"像素相似度"更接近人类观感。

## 4. 多模态与视频：一条路上的两个延伸

- **多模态理解**：Qwen-VL / GPT-4V——ViT 视觉编码 + LLM 生成，图转文、图文问答、图表理解；
- **视频生成**：Sora / Kling——把扩散的"空间 patch"扩成"时空 patch"，加时间注意力；本质是"图像扩散 + 时间一致"。

## 5. 全景图（识别 ↔ 生成）

```
识别线:   监督CNN → ViT → 自监督(MAE/CLIP) → 多模态
生成线:   GAN → VAE → 扩散 → 流/一致性 → 视频/3D
                ↓ 合流 ↓
        多模态生成(文生图/文生视频) + 可控生成(LoRA/ControlNet)
```

**本模块回顾**：01 ViT 是识别骨干；02 迁移/增强是数据效率；03 检测分割是空间任务；04-06 是生成三大范式；07 是文生图合流；08 是评估与展望。

## 课后练习

1. 把"坏生成器"改成"复制真实样本的子集"（多样性缺失但单张逼真），算 FID——为什么还是很高？这正对应"模式坍缩检测"。
2. 思考：MAE 掩码 75% 与扩散加噪到 t≈T，有什么相似之处？（提示：都是"毁掉输入再学会恢复"）
3. 查阅 SimCLR 的 InfoNCE 损失公式，说明它和交叉熵的关系。